# 05 — Metrics deep dive

Every metric the project reports, derived from scratch and verified against the project's own `ccdp.eval.metrics` implementations.

## Classification (per-class)
- **Precision** = TP / (TP + FP)
- **Recall**    = TP / (TP + FN)
- **F1**        = harmonic mean of precision and recall
- **Support**   = number of true examples of this class

## Regression (cost)
- **RMSE** — root mean squared error
- **MAE**  — mean absolute error
- **MAPE** — mean absolute percentage error
- **R²**   — coefficient of determination

## Detection
- **IoU**  (already covered in NB 03)
- **mAP** (already covered in NB 03)


In [ ]:
# === Colab bootstrap ===
# Safe to re-run. On a local clone with `pip install -e .` already done this
# is a no-op; on Colab it clones the repo + installs deps the first time.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/theDocWho/car-crash-fix-amount-predictor.git"
REPO_DIR = Path("car-crash-fix-amount-predictor")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)

# Make `ccdp` importable whether or not the package was installed editable.
src_path = Path("src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("ccdp path:", src_path)
print("running in Colab:", IN_COLAB)


## Precision, Recall, F1 — derivation

For one class, build a 2×2 *confusion matrix*:

|             | predicted positive | predicted negative |
|------------:|:------------------:|:------------------:|
| true pos    | **TP**             | FN                 |
| true neg    | FP                 | TN                 |

$$
\text{precision} = \frac{TP}{TP + FP}, \quad \text{recall} = \frac{TP}{TP + FN}
$$

Precision answers "**when the model said yes, was it right?**"
Recall answers "**of all the actual positives, how many did we catch?**"

F1 is their harmonic mean — it punishes models that get one of them very low:

$$
F_1 = 2 \cdot \frac{P \cdot R}{P + R}
$$


In [ ]:
import numpy as np

# 6 examples of class "dent" — model probabilities and ground truth.
# (we'll threshold at 0.5)
prob   = np.array([0.9, 0.8, 0.4, 0.2, 0.7, 0.1])
truth  = np.array([1,   1,   1,   0,   0,   0])
pred   = (prob >= 0.5).astype(int)

tp = int(((pred==1) & (truth==1)).sum())
fp = int(((pred==1) & (truth==0)).sum())
fn = int(((pred==0) & (truth==1)).sum())

precision = tp / max(tp+fp, 1)
recall    = tp / max(tp+fn, 1)
f1        = 2 * precision * recall / max(precision+recall, 1e-9)

print(f"TP={tp} FP={fp} FN={fn}")
print(f"precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")


Compare to the project's own implementation:


In [ ]:
from ccdp.eval.metrics import per_class_prf
import numpy as np

# Reshape into the multi-label "[N, C]" form the function expects.
classes = ["dent"]
probs  = prob.reshape(-1, 1)
labels = truth.reshape(-1, 1).astype(float)
m = per_class_prf(probs, labels, classes)
print(m["per_class"]["dent"])


## Regression metrics — derivation


In [ ]:
import numpy as np

y_true = np.array([100, 200, 300, 400], dtype=float)
y_pred = np.array([110, 180, 330, 360], dtype=float)
err = y_pred - y_true

rmse = np.sqrt((err**2).mean())
mae  = np.abs(err).mean()
mape = (np.abs(err) / np.abs(y_true)).mean() * 100
# R² = 1 − SS_res / SS_tot
ss_res = (err**2).sum()
ss_tot = ((y_true - y_true.mean())**2).sum()
r2 = 1 - ss_res / ss_tot

print(f"RMSE = {rmse:.2f}")
print(f"MAE  = {mae:.2f}")
print(f"MAPE = {mape:.2f} %")
print(f"R²   = {r2:.4f}")


**Which one should I report?** Insurance triage is most interpretable in **MAE** (dollars off, on average) and **MAPE** (% off). RMSE is more sensitive to outliers — good for catching the model when it occasionally predicts $5 instead of $500.

Verify against the project:


In [ ]:
from ccdp.eval.metrics import regression_metrics
m = regression_metrics(y_true.tolist(), y_pred.tolist())
m


## Why F1 instead of just accuracy?

If 95% of images have NO dent, a model that always says "no dent" gets 95% accuracy but is useless. Precision/recall/F1 ignore the easy negatives and only look at how well the model handles positives. **Use F1 (or precision-recall AUC) for any class-imbalanced problem.**
